<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_1/notebooks/Describing_Olig2_Ki67_data__STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Describing Your Data — Olig2⁺Ki67⁺ Density in CON and DRUG

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**

## The scenario

You've just taken over the analysis for your lab. Forty images have been through the Unit 1 pipeline — cortex and corpus callosum sections from control animals (**CON**) and drug-treated animals (**DRUG**), stained for **Olig2** (AF488, oligodendrocyte lineage) and **Ki67** (AF647, actively dividing).

Cells positive for **both** markers are *proliferating oligodendrocyte-lineage cells*. Their density, `Colocalized_cells_per_mm2`, is the readout for the lab's question:

> **Does the drug increase proliferation of oligodendrocyte-lineage cells?**

Your supervisor wants a one-paragraph summary by Friday. You have a folder of numbers and no idea yet what they look like. These eight steps take you from raw column to defensible claim.

| | Step | |
|---|---|---|
| **Part 1** | 1 | Build a histogram |
| | 2 | Compare the two groups' histograms — 💬 **Discussion 1** |
| | 3 | Mean, median, and mode of both groups |
| | 4 | Is the data positively or negatively skewed? — 💬 **Discussion 2** |
| **Part 2** | 5 | The interquartile range |
| | 6 | Sample variance |
| | 7 | Standard deviation — 💬 **Discussion 3** |
| | 8 | Final plot and the write-up — 💬 **Discussion 4** |

### How this notebook works

**Run every cell in order** (Shift + Enter). 💬 **Discussion** = double-click and type your answer. 🔬 **Research note** = the methods point behind the statistics.

**Submitting** — **Name:** *(double-click to type)*  **Date:** *(double-click to type)*

Answer all four 💬 Discussion cells → **Runtime → Run all** → **File → Print → Save as PDF** → upload to Canvas.

**Symbols:** $X$ = one score · $\sum X$ = sum of scores · $n$ = number of scores in a sample · $\bar{X}$ = sample mean · $s^2$ = sample variance · $s$ = sample standard deviation

---
# Part 1 — What shape is the data, and where is its centre?
---

## Step 1 — Build a histogram

Before computing a single statistic, **look at your data**. A histogram sorts the values into bins and shows how many fall into each — it's the fastest way to see the shape of a distribution.

Let's start with the control group alone.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_1/data/"
images = pd.read_csv(DATA + "images.csv")
print("Loaded", len(images), "images.\n")

con = images[images["Group"] == "CON"]["Colocalized_cells_per_mm2"]
drug = images[images["Group"] == "DRUG"]["Colocalized_cells_per_mm2"]
print("CON:", len(con), "images   DRUG:", len(drug), "images")

BINS = np.arange(0, 45, 5)        # bins 5 units wide, from 0 to 40

plt.figure(figsize=(6, 4))
plt.hist(con, bins=BINS, color="gray", edgecolor="black")
plt.xlabel("Olig2+Ki67+ cells per mm²")
plt.ylabel("Number of images")
plt.title("CON group (n = 20)")
plt.show()

counts, edges = np.histogram(con, bins=BINS)
print("\nImages per bin:", counts)

## Step 2 — Compare the two histograms

One histogram tells you about one group. The comparison is the experiment. Plotting both on the **same axes and the same bins** is essential — different bins would make the two groups look different even if they weren't.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, sharey=True)

for ax, (name, data, colour) in zip(axes, [("CON", con, "gray"),
                                           ("DRUG", drug, "tomato")]):
    ax.hist(data, bins=BINS, color=colour, edgecolor="black")
    ax.set_title(name + " (n = " + str(len(data)) + ")")
    ax.set_xlabel("Olig2+Ki67+ cells per mm²")
axes[0].set_ylabel("Number of images")
plt.tight_layout(); plt.show()

print("Images per bin, same bins for both groups:")
print("  Bin edges:", BINS)
print("  CON: ", np.histogram(con, bins=BINS)[0])
print("  DRUG:", np.histogram(drug, bins=BINS)[0])

### 💬 Discussion 1 — Reading the two distributions

1. Describe the difference between the two histograms in your own words. Comment on **where each is centred** and **how wide each is**.
2. Do the two distributions **overlap**? If you were handed a single unlabelled image with a density of 12 cells/mm², could you say confidently which group it came from?
3. 🔬 Why did we force both plots to use the same bins and the same axes? What could go wrong in a figure where two groups are plotted with different bin widths?

**Your answer:** *(double-click here to type)*

>


## Step 3 — Mean, median, and mode

Now put numbers on where each distribution sits. Three candidates for "the typical value":

- **Mean** $\bar{X} = \frac{\sum X}{n}$ — the average, and the balance point of the distribution
- **Median** — the middle value once the data are sorted
- **Mode** — the most common value. For a continuous measurement like density, no two images have exactly the same value, so we take the **midpoint of the tallest histogram bar**.

In [ ]:
def mode_from_histogram(x, bins):
    "For continuous data the mode is the middle of the tallest bar."
    heights, edges = np.histogram(x, bins=bins)
    t = np.argmax(heights)
    return (edges[t] + edges[t + 1]) / 2

summary = pd.DataFrame({
    "Mean": [con.mean(), drug.mean()],
    "Median": [con.median(), drug.median()],
    "Mode": [mode_from_histogram(con, BINS), mode_from_histogram(drug, BINS)],
}, index=["CON", "DRUG"]).round(2)

print("Olig2+Ki67+ cells per mm²\n")
print(summary)
print("\nDRUG mean / CON mean =", round(drug.mean() / con.mean(), 2))

## Step 4 — Positive or negative skew?

A distribution is **skewed** when one tail is longer than the other, and the direction is named for **the tail**:

- **Positive skew** — long tail to the **right**. The mean is pulled up: **mode < median < mean**
- **Negative skew** — long tail to the **left**. The mean is pulled down: **mean < median < mode**
- **Symmetric** — no long tail, and all three land in about the same place

So the *order* of the three numbers you just computed tells you the shape without needing to look at the plot. Let's check that against the plot.

In [ ]:
for name, data in [("CON", con), ("DRUG", drug)]:
    mean, median = data.mean(), data.median()
    mode = mode_from_histogram(data, BINS)
    shape = "POSITIVE skew" if mean > median else "NEGATIVE skew"
    print(name, "| mode", round(mode, 2), "< median", round(median, 2),
          "< mean", round(mean, 2), " ->", shape)
    print("     mean - median =", round(mean - median, 2), "\n")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
for ax, (name, data, colour) in zip(axes, [("CON", con, "gray"),
                                           ("DRUG", drug, "tomato")]):
    ax.hist(data, bins=BINS, color=colour, edgecolor="white")
    ax.axvline(mode_from_histogram(data, BINS), color="green", lw=2.5, label="Mode")
    ax.axvline(data.median(), color="blue", lw=2.5, label="Median")
    ax.axvline(data.mean(), color="red", lw=2.5, label="Mean")
    ax.set_title(name); ax.set_xlabel("Olig2+Ki67+ cells per mm²"); ax.legend()
axes[0].set_ylabel("Number of images")
plt.tight_layout(); plt.show()

### 💬 Discussion 2 — Skew, and which measure to report

1. Which direction is each group skewed? Write the three measures in order for each group, and say which group is skewed **more strongly**.
2. Explain *why* the mean ends up on the tail side of the median. (Think about what the mean does with a very large value that the median does not.)
3. 🔬 Density is a count per unit area, so it **cannot go below zero**, but nothing caps it from above. Explain why that alone makes positive skew the expected shape for this kind of measurement — and name one other readout from your Unit 1 pipeline you'd expect to behave the same way.
4. Your supervisor wants **one** number for the typical DRUG value. Which do you give, and how would you justify it?

**Your answer:** *(double-click here to type)*

>


---
### End of Part 1

You now know the **shape** of both distributions (positively skewed, DRUG more so) and where each is **centred** (DRUG about 1.9× CON). The histograms also showed something the centres can't capture: the DRUG group is far more **spread out**. Part 2 puts numbers on that.

---

# Part 2 — How spread out is it, and what may I claim?
---

## Step 5 — The interquartile range

The **IQR** is the range covered by the middle 50% of the data. It throws away the top and bottom quarters before measuring, which makes it stable when a few values are extreme.

To find it by hand:

1. Sort the data
2. Split it in half at the median
3. **Q1** = median of the lower half, **Q3** = median of the upper half
4. $IQR = Q3 - Q1$

In [ ]:
def quartiles_by_hand(x):
    "Split the sorted data at the median, then take the median of each half."
    x = np.sort(x)
    half = len(x) // 2
    lower = x[:half]
    upper = x[half:] if len(x) % 2 == 0 else x[half + 1:]   # odd n: skip the median
    return np.median(lower), np.median(upper)

for name, data in [("CON ", con), ("DRUG", drug)]:
    q1, q3 = quartiles_by_hand(data.values)
    print(name, "| Q1:", round(q1, 2), " median:", round(data.median(), 2),
          " Q3:", round(q3, 2), " -> IQR:", round(q3 - q1, 2))
    print("      full range:", round(data.min(), 2), "to", round(data.max(), 2),
          " =", round(data.max() - data.min(), 2), "\n")

## Step 6 — Sample variance

The IQR uses two numbers. **Variance** uses every score: on average, how far is each score from the mean?

The differences from the mean always add up to zero, so we **square** them first — that makes everything positive, so distances below and above the mean can't cancel out.

Our 20 images per group are a **sample** (we want to say something about the drug in general, not just these sections), so we divide by $n - 1$:

$$ s^2 = \frac{\sum (X - \bar{X})^2}{n - 1} $$

In `numpy` this is `ddof=1`. **Watch out:** `np.var()` defaults to `ddof=0`, which is the population formula and the wrong one for experimental data.

In [ ]:
# The long way, for CON — exactly as you'd do it on paper
work = pd.DataFrame({"Score (X)": con.values})
work["Difference"] = work["Score (X)"] - con.mean()
work["Difference squared"] = work["Difference"] ** 2

ss, n = work["Difference squared"].sum(), len(con)
print("CON mean =", round(con.mean(), 2))
print("The differences add up to:", round(work["Difference"].sum(), 10))
print("Sum of squared differences =", round(ss, 2))
print("Sample variance = ", round(ss, 2), "/", n - 1, "=", round(ss / (n - 1), 2), "\n")

print("First five rows of the working:")
print(work.head().round(2), "\n")

# The short way, for both groups
print("CON  sample variance:", round(con.var(ddof=1), 2), " (cells per mm² SQUARED)")
print("DRUG sample variance:", round(drug.var(ddof=1), 2), " (cells per mm² SQUARED)")

## Step 7 — Standard deviation

Variance has one practical problem: it's in **squared units**. "76 cells per mm² squared" means nothing to a reader.

The fix is to take the square root, giving the **standard deviation** — roughly the typical distance between a score and the mean, back in the original units:

$$ s = \sqrt{s^2} $$

Because it shares the units of the data, the SD can be read straight off the same axis as the mean — and a range of mean ± SD gives the reader a sense of where a typical image falls.

In [ ]:
for name, data in [("CON ", con), ("DRUG", drug)]:
    m, sd = data.mean(), data.std(ddof=1)
    print(name, "| variance:", round(data.var(ddof=1), 2), " SD:", round(sd, 2))
    print("      as a fraction of the mean:", round(sd / m * 100), "%\n")

print("DRUG SD / CON SD =", round(drug.std(ddof=1) / con.std(ddof=1), 2))

### 💬 Discussion 3 — Putting a number on the spread

1. Compare the two groups on **IQR** and on **SD**. Do both measures agree about which group is more variable, and roughly by how much?
2. Why did we square the differences before averaging them in Step 6 — and why did we then take a square root in Step 7?
3. We used `ddof=1` throughout. What would change if we had used the default `ddof=0`, and why is `ddof=1` the right choice for these 20 images?
4. 🔬 In Step 4 you found both groups are positively skewed, DRUG more strongly. Given that, which pair of numbers should you report for the DRUG group — the **mean and SD**, or the **median and IQR**? Justify your choice.

**Your answer:** *(double-click here to type)*

>


## Step 8 — The final plot, and the write-up

A **box plot** shows the median, the IQR, and the extremes at once — everything you computed in Steps 5 to 7, in one figure. The box covers the middle 50%, the line inside it is the median, the whiskers reach out to the rest of the data, and separate dots are unusually extreme values.

Plotting the individual images on top is good practice at this sample size: with n = 20 there's no reason to hide the raw data behind a summary. The right-hand panel shows the *same data* the way it is most often drawn in papers — two bars, two means. Compare them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Left: box plot with every image shown on top
axes[0].boxplot([con, drug], widths=0.5)
for i, data in enumerate([con, drug], start=1):
    jitter = np.random.default_rng(0).normal(i, 0.06, len(data))
    axes[0].scatter(jitter, data, color="black", alpha=0.6, zorder=3)
axes[0].set_xticks([1, 2]); axes[0].set_xticklabels(["CON", "DRUG"])
axes[0].set_ylabel("Olig2+Ki67+ cells per mm²")
axes[0].set_title("Every image, with median and IQR")

# Right: the same data as a plain bar chart of the two means
axes[1].bar([1, 2], [con.mean(), drug.mean()],
            color=["gray", "tomato"], edgecolor="black")
axes[1].set_xticks([1, 2]); axes[1].set_xticklabels(["CON", "DRUG"])
axes[1].set_ylabel("Olig2+Ki67+ cells per mm²")
axes[1].set_title("Group means only")
plt.tight_layout(); plt.show()

final = pd.DataFrame({
    "n": [len(con), len(drug)],
    "Mean": [con.mean(), drug.mean()],
    "Median": [con.median(), drug.median()],
    "SD": [con.std(ddof=1), drug.std(ddof=1)],
    "IQR": [np.diff(quartiles_by_hand(con.values))[0],
            np.diff(quartiles_by_hand(drug.values))[0]],
    "Min": [con.min(), drug.min()],
    "Max": [con.max(), drug.max()],
}, index=["CON", "DRUG"]).round(2)
print(final)

### 💬 Discussion 4 — What can you tell your supervisor?

1. Compare the two panels. What does the box plot show that the bar chart hides? Which would you put in a paper, and why?
2. Did the drug simply **shift** the values up, or did it also **spread them out**? What might that mean biologically?
3. 🔬 Write the paragraph. Three or four sentences you could send your supervisor: what the comparison shows (with numbers, units, and n), what the spread told you that the averages alone could not, and one limitation you must acknowledge.
4. 🔬 Now be the reviewer. Read your own paragraph back, and write the single hardest question you would ask its author.

**Your answer:** *(double-click here to type)*

>


## Summary — the eight steps

| Step | What it told you | CON | DRUG |
|---|---|---|---|
| 1–2 | **Histogram** — the shape, and the overlap between groups | compact, low | wide, shifted right |
| 3 | **Mean / median / mode** — where each group sits | 8.63 / 8.46 / 7.5 | 16.66 / 15.56 / 12.5 |
| 4 | **Skew** — mode < median < mean in both groups | positive (slight) | positive (stronger) |
| 5 | **IQR** — spread of the middle 50% | 5.76 | 10.44 |
| 6 | **Sample variance** ($ddof=1$) — in squared units | 12.55 | 76.46 |
| 7 | **Standard deviation** — back in original units | 3.54 | 8.74 |
| 8 | **Box plot + write-up** — what may be claimed | — | — |

**Five things to take away**

- **Look before you calculate.** The histograms in Steps 1–2 showed the shift, the width, and the overlap before a single statistic was computed.
- **The mean chases the tail.** The order mode < median < mean *is* the diagnosis of positive skew.
- **The centre and the spread go together.** A mean without a measure of spread is an incomplete result — here, the drug changed both.
- **Match the statistic to the shape.** Mean with SD, or median with IQR — they are matched pairs, and the skew you found in Step 4 decides which pair is honest.
- **Variability is a result, not just noise.** The doubled spread in the DRUG group is a finding in its own right: the response was not uniform across sections.

**And the limit of everything here:** every number in this notebook is *descriptive*. You have characterised these 40 images. You cannot yet say whether a difference this large is more than sampling variation would produce — that question needs inference, and it's where the next unit begins.

---
### Nice work — you're done!

Answer all four 💬 Discussion cells → **Runtime → Run all** → **File → Print → Save as PDF** → upload to Canvas.